# Módulo 12 — Despliegue de un modelo de clasificación en Vercel
### ¿Lloverá mañana? De la regresión logística a una app pública

En la práctica anterior desplegamos la **regresión lineal** (valor de vivienda). Ahora haremos lo mismo con un modelo de **clasificación**: la **regresión logística** del Módulo 8, que estima la probabilidad de lluvia a partir de la temperatura, la humedad y el viento.

El recorrido es el mismo:

```
 Colab                 Archivo            Código              GitHub           Vercel
 ─────                 ───────            ──────              ──────           ──────
 entrenar modelo  ──▶  modelo.json  ──▶  API (FastAPI)  ──▶  repositorio ──▶  URL pública
                                         + página web
```

**Lo nuevo respecto a la regresión:**

| | Regresión lineal | Regresión logística |
|---|---|---|
| Salida del modelo | Un número (valor de la vivienda) | Una probabilidad entre 0 y 1 |
| Ecuación | $\hat{y} = Xw + b$ | $p = \sigma(Xw + b)$ |
| Decisión final | No hay: el número es la respuesta | Se compara $p$ con un **umbral** → Sí / No |
| Qué guarda el JSON | μ, σ, pesos, sesgo | pesos, sesgo y **umbral** |

**Contenido:**
1. Generar los datos (los mismos del Módulo 8)
2. Entrenar y evaluar el modelo
3. Exportar el modelo a `modelo.json`
4. Verificar que el JSON clasifica igual que scikit-learn
5. Construir la app
6. Probarla antes de desplegar
7. Descargar los archivos
8. Desplegar en Vercel
9. Retos

## 0. Preparar el entorno

In [ ]:
!pip install -q fastapi uvicorn httpx

import json
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay

print("Listo ✔")

## 1. Los datos: los mismos días simulados del Módulo 8

Generamos el mismo conjunto de 300 días con la misma semilla, así que los datos son idénticos a los de aquella práctica.

| Variable | Significado |
|---|---|
| `temperatura` | Temperatura promedio del día (°C) |
| `humedad` | Humedad relativa (%) |
| `viento` | Velocidad del viento (km/h) |
| `llovio` | 1 = llovió, 0 = no llovió |

In [ ]:
np.random.seed(42)
n = 300

temperatura = np.random.normal(24, 5, n)
humedad = np.random.normal(65, 15, n)
viento = np.random.normal(10, 5, n)

# La probabilidad "real" de lluvia depende sobre todo de la humedad
z = -6 + 0.09 * humedad - 0.05 * temperatura + 0.02 * viento
prob_real = 1 / (1 + np.exp(-z))
llovio = (np.random.rand(n) < prob_real).astype(int)

df = pd.DataFrame({
    "temperatura": temperatura.round(1),
    "humedad": humedad.round(1),
    "viento": viento.round(1),
    "llovio": llovio,
})

print(f"Días: {len(df)} | Con lluvia: {df.llovio.sum()} | Sin lluvia: {(df.llovio == 0).sum()}")
df.head()

## 2. Entrenar y evaluar el modelo (repaso del Módulo 8)

In [ ]:
variables = ["temperatura", "humedad", "viento"]
X = df[variables]
y = df["llovio"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

modelo = LogisticRegression()
modelo.fit(X_train, y_train)

umbral = 0.5
prob_test = modelo.predict_proba(X_test)[:, 1]
pred_test = (prob_test >= umbral).astype(int)

exactitud = accuracy_score(y_test, pred_test)
precision = precision_score(y_test, pred_test)
recall = recall_score(y_test, pred_test)

print(f"Accuracy:  {exactitud:.2%}")
print(f"Precisión: {precision:.2%}")
print(f"Recall:    {recall:.2%}")

ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=["No llueve", "Sí llueve"],
                                        cmap="Blues", colorbar=False)
plt.title("Matriz de confusión (prueba)")
plt.show()

## 3. ¿Qué es este modelo entrenado?

Igual que la regresión lineal, todo lo que la regresión logística aprendió cabe en unos pocos números:

In [ ]:
pd.DataFrame({"variable": variables, "peso (w)": modelo.coef_[0].round(4)}).set_index("variable")

In [ ]:
print(f"Sesgo (b): {modelo.intercept_[0]:.4f}")

Para clasificar un día nuevo se hacen tres pasos:

$$z = b + w_1 \cdot \text{temperatura} + w_2 \cdot \text{humedad} + w_3 \cdot \text{viento}$$

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$

$$\text{decisión} = \begin{cases} \text{Sí lloverá} & \text{si } p \geq \text{umbral} \\ \text{No lloverá} & \text{si } p < \text{umbral} \end{cases}$$

El primer paso es exactamente la regresión lineal. La sigmoide convierte ese número en una probabilidad, y el umbral la convierte en una decisión.

> 💡 **¿Y la normalización?** En el Módulo 8 entrenamos este modelo con los datos originales, sin normalizar. Por eso aquí **no** hay que guardar medias ni desviaciones. La regla es siempre la misma: el JSON debe llevar **todo lo que el modelo necesita para predecir**, ni más ni menos.

El **umbral** no lo aprende el modelo: es una decisión nuestra. Por eso es útil guardarlo en el JSON: si mañana queremos un modelo más "cauteloso", cambiamos el umbral sin reentrenar.

In [ ]:
modelo_json = {
    "nombre": "Regresión logística — ¿lloverá mañana?",
    "variables": variables,
    "pesos": modelo.coef_[0].tolist(),
    "sesgo": float(modelo.intercept_[0]),
    "umbral": umbral,
    "metricas_prueba": {
        "accuracy": round(float(exactitud), 4),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
    },
    "clases": {"0": "No lloverá", "1": "Sí lloverá"},
}

Path("modelo.json").write_text(json.dumps(modelo_json, indent=2, ensure_ascii=False), encoding="utf-8")

print(Path("modelo.json").read_text())
print(f"Tamaño del archivo: {Path('modelo.json').stat().st_size} bytes")

## 4. Verificar: ¿el JSON clasifica igual que scikit-learn?

Esta es la función que usará la app. Solo usa la librería `math` de Python: nada de NumPy ni scikit-learn.

In [ ]:
def sigmoide(z):
    # Versión estable: evita desbordamientos con valores de z muy grandes o muy pequeños
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    e = math.exp(z)
    return e / (1 + e)


def clasificar(datos, modelo):
    z = modelo["sesgo"]
    for nombre, w in zip(modelo["variables"], modelo["pesos"]):
        z += w * datos[nombre]
    p = sigmoide(z)
    return p, int(p >= modelo["umbral"])


modelo_cargado = json.loads(Path("modelo.json").read_text())

muestras = X_test.head(6)
resultados = [clasificar(fila, modelo_cargado) for fila in muestras.to_dict("records")]

comparacion = pd.DataFrame({
    "p scikit-learn": modelo.predict_proba(muestras)[:, 1],
    "p JSON (Python puro)": [p for p, _ in resultados],
    "clase JSON": [c for _, c in resultados],
    "clase real": y_test.head(6).values,
}).round(4)

display(comparacion)
assert np.allclose(comparacion.iloc[:, 0], comparacion.iloc[:, 1]), "¡No coinciden!"
print("✔ El modelo exportado da las mismas probabilidades")

## 5. Construir la app

Los mismos cuatro archivos de la práctica anterior:

| Archivo | Qué hace |
|---|---|
| `main.py` | La **API** (backend): carga `modelo.json` y clasifica |
| `index.html` | La **página** (frontend): formulario con deslizadores |
| `modelo.json` | El modelo entrenado |
| `requirements.txt` | Las librerías del servidor |

In [ ]:
carpeta_app = Path("app-lluvia")
carpeta_app.mkdir(exist_ok=True)
shutil.copy("modelo.json", carpeta_app / "modelo.json")
print("Carpeta creada:", carpeta_app.resolve())

### 5.1 La API: `main.py`

Tiene las mismas tres rutas. Hay una mejora respecto a la app de vivienda: la clase `Dia` ahora **valida rangos**. `ge` significa "mayor o igual que" y `le`, "menor o igual que". Si alguien envía una humedad de 150 %, FastAPI la rechaza con un error 422 antes de llegar al modelo.

In [ ]:
%%writefile app-lluvia/main.py
"""
Módulo 12 — API del modelo de regresión logística (¿lloverá mañana?)

Rutas:
    GET  /          → página con el formulario
    GET  /modelo    → información y métricas del modelo
    POST /predict   → probabilidad de lluvia y decisión
    GET  /docs      → documentación automática (la genera FastAPI)
"""
import json
import math
from pathlib import Path

from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

CARPETA = Path(__file__).parent
MODELO = json.loads((CARPETA / "modelo.json").read_text(encoding="utf-8"))

app = FastAPI(
    title="¿Lloverá mañana?",
    description="Regresión logística del Módulo 8, desplegada en el Módulo 12.",
    version="1.0",
)


class Dia(BaseModel):
    temperatura: float = Field(..., ge=-10, le=50, description="Temperatura promedio (°C)", examples=[24])
    humedad: float = Field(..., ge=0, le=100, description="Humedad relativa (%)", examples=[85])
    viento: float = Field(..., ge=0, le=150, description="Velocidad del viento (km/h)", examples=[12])


def sigmoide(z: float) -> float:
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    e = math.exp(z)
    return e / (1 + e)


def clasificar(datos: dict):
    # z = b + Σ w_i · x_i   →   p = σ(z)   →   clase = p ≥ umbral
    z = MODELO["sesgo"]
    for nombre, w in zip(MODELO["variables"], MODELO["pesos"]):
        z += w * datos[nombre]
    p = sigmoide(z)
    return p, int(p >= MODELO["umbral"])


@app.get("/", response_class=HTMLResponse, include_in_schema=False)
def pagina():
    return (CARPETA / "index.html").read_text(encoding="utf-8")


@app.get("/modelo")
def info_modelo():
    return {
        "nombre": MODELO["nombre"],
        "variables": MODELO["variables"],
        "umbral": MODELO["umbral"],
        "metricas_prueba": MODELO["metricas_prueba"],
    }


@app.post("/predict")
def predict(dia: Dia):
    p, clase = clasificar(dia.model_dump())
    return {
        "probabilidad": round(p, 4),
        "clase": clase,
        "decision": MODELO["clases"][str(clase)],
        "umbral": MODELO["umbral"],
    }

### 5.2 La página: `index.html`

En lugar de cajas de texto usamos **deslizadores** (`<input type="range">`), que impiden escribir valores fuera de rango. El resultado muestra la probabilidad como una barra, con una marca donde está el umbral: así se ve por qué el modelo decidió "Sí" o "No".

In [ ]:
%%writefile app-lluvia/index.html
<!doctype html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>¿Lloverá mañana?</title>
  <style>
    :root { --fondo:#f2f5f9; --tarjeta:#fff; --texto:#1b2633; --suave:#5d6a79; --acento:#2f5fa7; --borde:#d7dee7;
            --lluvia:#2f5fa7; --seco:#c98a12; }
    * { box-sizing: border-box; }
    body { margin:0; font-family: system-ui, -apple-system, "Segoe UI", sans-serif; background:var(--fondo); color:var(--texto); }
    main { max-width: 640px; margin: 0 auto; padding: 32px 16px; }
    h1 { margin: 0 0 4px; font-size: 1.6rem; }
    .sub { color: var(--suave); margin: 0 0 24px; }
    .tarjeta { background: var(--tarjeta); border: 1px solid var(--borde); border-radius: 12px; padding: 24px; }
    .campo { margin-bottom: 22px; }
    .fila { display: flex; justify-content: space-between; align-items: baseline; margin-bottom: 8px; }
    label { font-weight: 600; }
    .valor { font-variant-numeric: tabular-nums; color: var(--acento); font-weight: 700; }
    input[type=range] { width: 100%; accent-color: var(--acento); }
    button { width: 100%; padding: 12px; font-size: 1rem; font-weight: 600; color: #fff;
             background: var(--acento); border: 0; border-radius: 8px; cursor: pointer; }
    button:disabled { opacity: .6; }
    #resultado { margin-top: 22px; display: none; }
    #decision { font-size: 1.8rem; font-weight: 700; margin-bottom: 12px; }
    .barra { position: relative; height: 18px; border-radius: 9px; background: #e4e9f0; overflow: visible; }
    .relleno { height: 100%; border-radius: 9px; width: 0; transition: width .4s; }
    .marca { position: absolute; top: -6px; width: 2px; height: 30px; background: var(--texto); }
    .marca span { position: absolute; top: 32px; left: 50%; transform: translateX(-50%);
                  font-size: .75rem; color: var(--suave); white-space: nowrap; }
    #detalle { margin-top: 30px; color: var(--suave); font-size: .9rem; }
    footer { margin-top: 20px; font-size: .85rem; color: var(--suave); }
    a { color: var(--acento); }
  </style>
</head>
<body>
<main>
  <h1>🌦️ ¿Lloverá mañana?</h1>
  <p class="sub">Regresión logística · probabilidad de lluvia según el clima del día</p>

  <form id="formulario" class="tarjeta">
    <div class="campo">
      <div class="fila"><label for="temperatura">Temperatura</label><span class="valor" id="v-temperatura"></span></div>
      <input id="temperatura" type="range" min="5" max="40" step="0.5" value="24" data-unidad=" °C">
    </div>
    <div class="campo">
      <div class="fila"><label for="humedad">Humedad</label><span class="valor" id="v-humedad"></span></div>
      <input id="humedad" type="range" min="10" max="100" step="1" value="85" data-unidad=" %">
    </div>
    <div class="campo">
      <div class="fila"><label for="viento">Viento</label><span class="valor" id="v-viento"></span></div>
      <input id="viento" type="range" min="0" max="40" step="1" value="12" data-unidad=" km/h">
    </div>

    <button id="boton" type="submit">Calcular probabilidad</button>

    <div id="resultado">
      <div id="decision"></div>
      <div class="barra">
        <div class="relleno" id="relleno"></div>
        <div class="marca" id="marca"><span id="texto-umbral"></span></div>
      </div>
      <div id="detalle"></div>
    </div>
  </form>

  <footer>
    <span id="info-modelo">Cargando información del modelo…</span> ·
    <a href="/docs">Ver la API (/docs)</a>
  </footer>
</main>

<script>
  const campos = ["temperatura", "humedad", "viento"];

  // Mostrar el valor de cada deslizador mientras se mueve
  for (const c of campos) {
    const control = document.getElementById(c);
    const mostrar = () => document.getElementById("v-" + c).textContent = control.value + control.dataset.unidad;
    control.addEventListener("input", mostrar);
    mostrar();
  }

  // Métricas del modelo desde la API
  fetch("/modelo").then(r => r.json()).then(m => {
    const k = m.metricas_prueba;
    document.getElementById("info-modelo").textContent =
      `Accuracy ${(k.accuracy * 100).toFixed(0)} % · Precisión ${(k.precision * 100).toFixed(0)} % · Recall ${(k.recall * 100).toFixed(0)} %`;
  });

  document.getElementById("formulario").addEventListener("submit", async (evento) => {
    evento.preventDefault();
    const boton = document.getElementById("boton");
    boton.disabled = true;

    // 1. Reunir los datos
    const datos = {};
    for (const c of campos) datos[c] = Number(document.getElementById(c).value);

    // 2. Enviarlos a la API
    try {
      const respuesta = await fetch("/predict", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify(datos),
      });
      if (!respuesta.ok) throw new Error(`Error ${respuesta.status}`);
      const r = await respuesta.json();

      // 3. Mostrar el resultado
      const pct = (r.probabilidad * 100).toFixed(1);
      const color = r.clase === 1 ? "var(--lluvia)" : "var(--seco)";
      document.getElementById("decision").textContent = (r.clase === 1 ? "☔ " : "☀️ ") + r.decision;
      document.getElementById("decision").style.color = color;
      document.getElementById("relleno").style.width = pct + "%";
      document.getElementById("relleno").style.background = color;
      document.getElementById("marca").style.left = (r.umbral * 100) + "%";
      document.getElementById("texto-umbral").textContent = "umbral " + r.umbral;
      document.getElementById("detalle").textContent =
        `Probabilidad de lluvia: ${pct} % · ${r.clase === 1 ? "supera" : "no alcanza"} el umbral de ${r.umbral}`;
    } catch (e) {
      document.getElementById("decision").textContent = "No se pudo calcular";
      document.getElementById("detalle").textContent = e.message;
    }
    document.getElementById("resultado").style.display = "block";
    boton.disabled = false;
  });
</script>
</body>
</html>

### 5.3 Dependencias y README

In [ ]:
%%writefile app-lluvia/requirements.txt
fastapi
uvicorn

In [ ]:
%%writefile app-lluvia/README.md
# ¿Lloverá mañana? — Módulo 12

Regresión logística (Módulo 8) desplegada como API con FastAPI en Vercel.

## Enlaces
- App: https://TU-PROYECTO.vercel.app
- Documentación de la API: https://TU-PROYECTO.vercel.app/docs

## Archivos
| Archivo | Rol |
|---|---|
| `modelo.json` | Pesos, sesgo, umbral y métricas del modelo |
| `main.py` | API: calcula la probabilidad en `/predict` |
| `index.html` | Página con los deslizadores |
| `requirements.txt` | Dependencias del servidor |

## Reflexión
(Escribe aquí tus respuestas a los retos del notebook)

## 6. Probar la app antes de desplegarla

In [ ]:
import sys, importlib
sys.path.insert(0, str(carpeta_app.resolve()))
import main
importlib.reload(main)

from fastapi.testclient import TestClient
cliente = TestClient(main.app)

print("GET /        →", cliente.get("/").status_code)
print("GET /modelo  →", cliente.get("/modelo").json())

for dia in [{"temperatura": 24, "humedad": 85, "viento": 12},
            {"temperatura": 30, "humedad": 40, "viento": 5}]:
    print("POST /predict", dia, "→", cliente.post("/predict", json=dia).json())

Ahora probamos la validación de rangos con una humedad imposible:

In [ ]:
r = cliente.post("/predict", json={"temperatura": 24, "humedad": 150, "viento": 12})
print("Código:", r.status_code)
print(r.json()["detail"][0]["msg"])

### (Opcional) Ver la página dentro de Colab

In [ ]:
import subprocess, time

servidor = subprocess.Popen(["uvicorn", "main:app", "--port", "8001"], cwd=carpeta_app)
time.sleep(3)

try:
    from google.colab import output
    output.serve_kernel_port_as_window(8001)
except ImportError:
    print("No estás en Colab: abre http://localhost:8001 en tu navegador")

## 7. Descargar los archivos

In [ ]:
# __pycache__ la crea Python al importar main.py en las pruebas: no se sube a GitHub
shutil.rmtree(carpeta_app / "__pycache__", ignore_errors=True)

shutil.make_archive("app-lluvia", "zip", carpeta_app)
import zipfile
print("Contenido del zip:", sorted(zipfile.ZipFile(f"{carpeta_app}.zip").namelist()))

try:
    from google.colab import files
    files.download("app-lluvia.zip")
except ImportError:
    print("Archivo listo: app-lluvia.zip")

## 8. Desplegar en Vercel

Es el mismo procedimiento de la app de vivienda, en un **repositorio nuevo**:

1. **GitHub:** crea un repositorio público llamado `app-lluvia` y sube los **5 archivos** del zip a la raíz (no la carpeta).
2. **Vercel:** *Add New… → Project*. Si `app-lluvia` no aparece en la lista, usa *Adjust GitHub App Permissions* para darle acceso a ese repositorio.
3. Presiona **Import**, verifica que el *Framework Preset* diga **FastAPI** y presiona **Deploy**.
4. Prueba la URL: `/`, `/docs` y `/modelo`.

### Si algo falla

| Síntoma | Causa probable |
|---|---|
| Vercel no detecta FastAPI | Los archivos están en una subcarpeta del repositorio |
| Error 500 al abrir la página | Falta `modelo.json` o `index.html`. Revisa la pestaña **Logs** |
| Error 422 en `/predict` | Un valor está fuera de rango o falta un campo |
| La barra no se mueve | Abre la consola del navegador (F12) y revisa la pestaña *Network* |

## 9. Retos

Responde en el `README.md` de tu repositorio:

1. **Cambia el umbral sin reentrenar.** En GitHub, edita `modelo.json` y cambia `"umbral": 0.5` por `0.7`. Espera el redespliegue y prueba el mismo día (24 °C, 85 %, 12 km/h). ¿Cambió la probabilidad? ¿Cambió la decisión? ¿Por qué?
2. **¿Por qué aquí no guardamos medias ni desviaciones**, y en la app de vivienda sí? ¿Qué pasaría si entrenaras este modelo con datos normalizados y olvidaras guardarlas?
3. **Encuentra la frontera.** Con temperatura y viento fijos, mueve la humedad hasta encontrar el punto donde la decisión cambia de "No" a "Sí". ¿Qué probabilidad muestra la barra en ese punto?
4. **Mira la red.** Abre las herramientas del navegador (F12 → *Network*), calcula una probabilidad y abre la petición `predict`. ¿Qué viaja en *Payload*? ¿Qué vuelve en *Response*?
5. **Compara las dos apps.** Llena una tabla con lo que cambió entre la app de vivienda y esta: la ecuación, el JSON, la respuesta de la API y la página. ¿Qué se quedó igual?

## Cierre

Con dos modelos distintos, una regresión y una clasificación, el despliegue fue **el mismo procedimiento**: exportar lo aprendido a un JSON, envolverlo en una API y publicarlo. Lo que cambia entre modelos es la función que calcula la predicción; la arquitectura se mantiene.

**Próxima sesión:** empaquetaremos estas apps en contenedores Docker.